In [ ]:
import pandas as pd
import numpy as np

# Load dữ liệu thô
df = pd.read_csv('../data/raw/parking_data_raw.csv')
df['entry_time'] = pd.to_datetime(df['entry_time'])
df['exit_time'] = pd.to_datetime(df['exit_time'])

print(f"Kích thước ban đầu: {df.shape}")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Kích thước ban đầu: (15000, 9)


In [2]:
# Tách các thành phần thời gian để model dễ học
df['entry_hour'] = df['entry_time'].dt.hour
df['entry_minute'] = df['entry_time'].dt.minute
df['day_of_week_num'] = df['entry_time'].dt.weekday # 0: Thứ 2, 6: Chủ nhật

# Cờ boolean (0 hoặc 1)
df['is_weekend'] = df['day_of_week_num'].isin([5, 6]).astype(int)
df['is_morning'] = (df['entry_hour'] < 12).astype(int)

df[['entry_time', 'entry_hour', 'is_weekend', 'is_morning']].head()

,entry_time,entry_hour,is_weekend,is_morning
0,2023-09-15 12:11:00,12,0,0
1,2023-09-03 07:09:00,7,1,1
2,2023-10-03 05:41:00,5,0,1
3,2023-09-15 07:28:00,7,0,1
4,2023-12-10 06:41:00,6,1,1


In [3]:
# Bắt buộc phải sort theo sinh viên và thời gian từ cũ đến mới
df = df.sort_values(by=['student_id', 'entry_time']).reset_index(drop=True)

# 1. Rolling Average: Thời gian đỗ trung bình 5 lần gần nhất của sinh viên đó
# (Dùng shift(1) để mô hình không bị "nhìn trộm" tương lai - Data Leakage)
df['rolling_avg_duration'] = df.groupby('student_id')['duration_minutes'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)
# Những lần gửi đầu tiên sẽ bị NaN, ta điền bằng mức trung bình toàn bãi
df['rolling_avg_duration'] = df['rolling_avg_duration'].fillna(df['duration_minutes'].mean())

# 2. Historical Overnight: Sinh viên này đã từng đỗ qua đêm bao nhiêu lần trước đây?
df['is_overnight_flag'] = (df['parking_behavior'] == 'Overnight').astype(int)
df['historical_overnight_count'] = df.groupby('student_id')['is_overnight_flag'].cumsum().shift(1).fillna(0)

# Xóa cột cờ tạm
df = df.drop(columns=['is_overnight_flag'])

df[['student_id', 'entry_time', 'duration_minutes', 'rolling_avg_duration', 'historical_overnight_count']].head(10)

,student_id,entry_time,duration_minutes,rolling_avg_duration,historical_overnight_count
0,SV0001,2023-09-16 06:35:00,357,485.490600,0.0
1,SV0001,2023-10-13 06:41:00,353,357.000000,0.0
2,SV0001,2023-11-10 06:53:00,378,355.000000,0.0
3,SV0001,2023-11-12 12:53:00,241,362.666667,0.0
4,SV0001,2023-12-20 13:11:00,134,332.250000,0.0
5,SV0002,2023-09-02 06:32:00,3296,485.490600,0.0
6,SV0002,2023-09-30 13:31:00,85,3296.000000,0.0
7,SV0002,2023-10-27 12:23:00,1500,1690.500000,0.0
8,SV0002,2023-10-29 06:50:00,108,1627.000000,0.0
9,SV0002,2023-11-05 06:01:00,284,1247.250000,0.0


In [4]:
# Encode loại xe và khu vực đỗ
df = pd.get_dummies(df, columns=['vehicle_type', 'usual_zone'], drop_first=True)

# Đảm bảo các cột dummy chuyển về dạng int thay vì boolean (True/False)
dummy_cols = [col for col in df.columns if 'vehicle_type_' in col or 'usual_zone_' in col]
df[dummy_cols] = df[dummy_cols].astype(int)

df.head()

,student_id,entry_time,exit_time,day_of_week,duration_minutes,is_exam_week,parking_behavior,entry_hour,entry_minute,day_of_week_num,is_weekend,is_morning,rolling_avg_duration,historical_overnight_count,vehicle_type_Motorbike,usual_zone_Zone_B,usual_zone_Zone_C,usual_zone_Zone_D
0,SV0001,2023-09-16 06:35:00,2023-09-16 12:32:53,Saturday,357,0,Full_Day,6,35,5,1,1,485.490600,0.0,1,0,1,0
1,SV0001,2023-10-13 06:41:00,2023-10-13 12:34:54,Friday,353,0,Full_Day,6,41,4,0,1,357.000000,0.0,1,1,0,0
2,SV0001,2023-11-10 06:53:00,2023-11-10 13:11:22,Friday,378,0,Full_Day,6,53,4,0,1,355.000000,0.0,1,1,0,0
3,SV0001,2023-11-12 12:53:00,2023-11-12 16:54:25,Sunday,241,0,Full_Day,12,53,6,1,0,362.666667,0.0,1,0,0,0
4,SV0001,2023-12-20 13:11:00,2023-12-20 15:25:28,Wednesday,134,1,Early_Return,13,11,2,0,0,332.250000,0.0,1,0,0,1


In [5]:
import os
os.makedirs('../data/processed', exist_ok=True)

# 1. Dataset cho bài toán Classification (Phân loại hành vi đỗ xe)
# Bỏ đi các cột không cần thiết hoặc gây data leakage (như exit_time)
drop_cols_clf = ['student_id', 'entry_time', 'exit_time', 'day_of_week', 'duration_minutes']
df_classification = df.drop(columns=drop_cols_clf)
df_classification.to_csv('../data/processed/classification_data.csv', index=False)

# 2. Dataset cho bài toán Regression (Dự đoán thời gian đỗ)
# Target ở đây là duration_minutes thay vì parking_behavior
drop_cols_reg = ['student_id', 'entry_time', 'exit_time', 'day_of_week', 'parking_behavior']
df_regression = df.drop(columns=drop_cols_reg)
df_regression.to_csv('../data/processed/regression_data.csv', index=False)

print("Đã xử lý xong Feature Engineering và lưu vào data/processed/")

Đã xử lý xong Feature Engineering và lưu vào data/processed/
